# Step 05. Do patient endotypes exist in cohort A?

An endotype is a set of patients sharing a coordinated shift across a set of proteins, a
rectangle in the matrix. This step asks whether cohort A contains any, and answers no.

**There is nothing in the source paper to check against.** Leung *et al.* never clustered patients;
they stratified by disease activity category and by autoantibody status. The module layer of this
pipeline reproduces their result (step 02). This step reproduces nothing, it is new, unreplicated
work, so the standard of evidence has to come from inside the data.

**Silhouette is not used here, and that is intentional.** An earlier version of this notebook chose
k by average silhouette width. Three things make it the wrong instrument, and all three bit:

1. It grades k-means with k-means' own assumptions. Silhouette rewards compact isotropic
   Euclidean blobs, exactly the structure k-means imposes whether or not the data has it. Scoring
   a k-means partition by silhouette asks the method to grade its own work.
2. It is not comparable across feature spaces. Its value depends on the dimension and on which
   modules are in it, so "0.265 on 8 modules beats 0.203 on 12" compares two geometries, not two
   partitions.
3. On this data it preferred a partition that dissolves. Its highest-scoring partition had a
   17-patient cluster with bootstrap Jaccard 0.49, below the line at which a cluster is considered
   not to exist.

It is replaced by two measures that can fail, and do.

In [1]:
set.seed(42)
source("../src/paths.R")
e   <- readRDS(art("eigengenes_A.rds")); sig <- readRDS(art("sig_modules_A.rds"))
w   <- readRDS(art("wgcna_A.rds"));      mods <- w$mods

# Two candidate spaces, because the choice of space is itself a result.
#   all      -- every clinically associated module
#   specific -- only those of 50 proteins or fewer. blue (1,213) and yellow (373)
#               are eigenproteins over a large fraction of the panel; in a
#               Euclidean distance they weigh the same as a 10-protein module.
sz    <- sapply(sig, function(k) sum(mods == k))
keep  <- sig[sz <= 50]
SPACE <- list(all      = scale(e$ME[, paste0("ME", sig),  drop = FALSE]),
              specific = scale(e$ME[, paste0("ME", keep), drop = FALSE]))
cat(sprintf("all      : %2d modules, %5d proteins\nspecific : %2d modules, %5d proteins\n",
            length(sig), sum(mods %in% sig), length(keep), sum(mods %in% keep)))
sort(sz)

all      : 12 modules,  1978 proteins
specific :  8 modules,   164 proteins


bisque4         brown4          ivory    darkmagenta darkolivegreen 
            10             10             14             19             21 
        violet       darkgrey    lightyellow    greenyellow          black 
            22             30             38             78            150 
        yellow           blue 
           373           1213

## Two measures that can fail

**Prediction strength** (Tibshirani & Walther 2005), *the* k-chooser. Split the patients in half,
cluster each half independently, then use the training half's centroids to predict which test
patients belong together. For each test cluster, score the fraction of its pairs the training
centroids also put together; the partition's score is the worst cluster's fraction. This is
cross-validation: the clustering has to work on patients it never saw. The published rule is to
take the largest k scoring above 0.8.

**Bootstrap Jaccard** (Hennig), per-cluster reappearance. Resample patients with replacement,
re-cluster, and record each original cluster's best Jaccard overlap with any bootstrap cluster.
This is what `fpc::clusterboot` packages; it is written out here because the calculation is the
argument.

| mean Jaccard | reading |
|---|---|
| < 0.5 | dissolved, not a real pattern |
| 0.6 – 0.75 | a pattern, but not a certain one |
| > 0.85 | highly stable |

Neither measures *separation*. Both measure whether the same patients would be grouped together
again, which is the claim an endotype actually makes.

In [2]:
KS <- 2:5
B_PS <- 100; B_BOOT <- 200

# Tibshirani & Walther prediction strength.
pred_strength <- function(S, K, B) {
  n <- nrow(S)
  vapply(seq_len(B), function(b) {
    i    <- sample(n, floor(n / 2))
    ctr  <- kmeans(S[i, , drop = FALSE],  K, nstart = 25)$centers
    cte  <- kmeans(S[-i, , drop = FALSE], K, nstart = 25)$cluster
    te   <- S[-i, , drop = FALSE]
    pred <- apply(te, 1, function(x) which.min(colSums((t(ctr) - x)^2)))
    min(vapply(seq_len(K), function(j) {
      m <- which(cte == j); nj <- length(m)
      if (nj < 2) return(NA_real_)
      (sum(outer(pred[m], pred[m], "==")) - nj) / (nj * (nj - 1))
    }, numeric(1)), na.rm = TRUE)
  }, numeric(1))
}

# Hennig cluster stability.
boot_jaccard <- function(S, K, B) {
  base <- kmeans(S, K, nstart = 50)$cluster; n <- nrow(S)
  J <- matrix(NA_real_, B, K)
  for (b in seq_len(B)) {
    idx <- sample(n, n, replace = TRUE)
    cb  <- tryCatch(kmeans(S[idx, , drop = FALSE], K, nstart = 25)$cluster,
                    error = function(e) NULL)
    if (is.null(cb)) next
    for (j in seq_len(K)) {
      orig <- which(base[idx] == j)
      if (!length(orig)) next
      J[b, j] <- max(vapply(seq_len(K), function(k) {
        bk <- which(cb == k); length(intersect(orig, bk)) / length(union(orig, bk))
      }, numeric(1)))
    }
  }
  colMeans(J, na.rm = TRUE)
}

PS_THRESHOLD <- 0.8
support <- do.call(rbind, lapply(names(SPACE), function(nm) {
  S <- SPACE[[nm]]
  do.call(rbind, lapply(KS, function(K) {
    set.seed(42); km  <- kmeans(S, K, nstart = 50)
    set.seed(42); ps  <- pred_strength(S, K, B_PS)
    set.seed(42); jac <- sort(boot_jaccard(S, K, B_BOOT), decreasing = TRUE)
    data.frame(space = nm, k = K, sizes = paste(sort(km$size), collapse = "/"),
               pred_strength = round(mean(ps), 2), ps_sd = round(sd(ps), 2),
               passes = mean(ps) > PS_THRESHOLD,
               min_jaccard = round(min(jac), 2),
               jaccard = paste(sprintf("%.2f", jac), collapse = " "))
  }))
}))
print(support, row.names = FALSE)
cat(sprintf("\npartitions passing prediction strength > %.1f: %d of %d\n",
            PS_THRESHOLD, sum(support$passes), nrow(support)))

    space k        sizes pred_strength ps_sd passes min_jaccard
      all 2        28/59          0.65  0.23  FALSE        0.64
      all 3      7/26/54          0.55  0.23  FALSE        0.65
      all 4    2/7/26/52          0.46  0.28  FALSE        0.50
      all 5  2/7/7/17/54          0.32  0.19  FALSE        0.56
 specific 2        17/70          0.69  0.18  FALSE        0.49
 specific 3      8/28/51          0.47  0.16  FALSE        0.56
 specific 4    2/9/34/42          0.40  0.17  FALSE        0.63
 specific 5 2/8/25/25/27          0.29  0.17  FALSE        0.62
                  jaccard
                0.80 0.64
           0.87 0.84 0.65
      0.79 0.79 0.68 0.50
 0.75 0.66 0.65 0.62 0.56
                0.81 0.49
           0.66 0.56 0.56
      0.67 0.66 0.66 0.63
 0.79 0.75 0.69 0.68 0.62



partitions passing prediction strength > 0.8: 0 of 8


## No partition passes

**No partition passes, at any k, in either space.** The best score anywhere is 0.69, two halves of
the same cohort, clustered independently, agree on which patients belong together about two thirds
of the time, where 0.8 is the floor for calling a partition real.

**Prediction strength falls monotonically with k.** 0.65 → 0.55 → 0.46 → 0.32 on the full space.
Data with genuine structure at some k shows a *peak* there. A monotonic decline from an already
failing k = 2 is the signature of no cluster structure at all: every added cluster is splitting
noise, and the split does not survive being shown new patients.

**Bootstrap Jaccard agrees, and shows what the failure looks like.** Individual clusters do reach
0.80–0.87, but they are always the large residual cluster, the one that reappears because it is
most of the cohort. The small clusters sit at 0.49–0.65: outliers that do not come back.

**So: zero supported endotypes in cohort A.** Not "two weak ones". Stated plainly:
87 patients on 12 module eigenproteins do not contain a partition that survives cross-validation,
and the right next step is not a better clustering algorithm.

The k = 2 partition is kept below as a description for the figure only, it is what k-means
returns when asked for two groups, and step 06 draws it so the absence of block structure is
visible rather than asserted. It is not a finding and is not saved as one.

In [3]:
# A DESCRIPTIVE partition, for the figure in step 06. Not a supported result --
# see the table above, where it scores 0.65 against a threshold of 0.8.
CHOSEN_SPACE <- "all"
S    <- SPACE[[CHOSEN_SPACE]]
set.seed(42)
K    <- 2
endo <- factor(paste0("E", kmeans(S, K, nstart = 50)$cluster))
stab <- support[support$space == CHOSEN_SPACE & support$k == K, ]
cat(sprintf("descriptive k = %d on %d modules, sizes %s\n  prediction strength %.2f (threshold %.1f: %s)\n  bootstrap Jaccard %s\n",
            K, ncol(S), paste(table(endo), collapse = " / "),
            stab$pred_strength, PS_THRESHOLD,
            if (stab$passes) "PASS" else "FAIL", stab$jaccard))
table(endo)

descriptive k = 2 on 12 modules, sizes 59 / 28
  prediction strength 0.65 (threshold 0.8: FAIL)
  bootstrap Jaccard 0.80 0.64


endo
E1 E2 
59 28 

In [4]:
# The block table: one row per descriptive group, one column per module.
Fs     <- apply(S, 2, function(x) summary(aov(x ~ endo))[[1]][["F value"]][1])
blocks <- t(sapply(levels(endo), function(g) colMeans(S[endo == g, , drop = FALSE])))
colnames(blocks) <- sub("^ME", "", colnames(blocks))
names(Fs)        <- sub("^ME", "", names(Fs))
round(blocks[, names(sort(Fs, decreasing = TRUE))], 2)

,blue,darkmagenta,black,bisque4,darkgrey,darkolivegreen,brown4,greenyellow,lightyellow,ivory,yellow,violet
E1,-0.49,-0.47,0.46,-0.35,-0.32,-0.25,0.11,0.09,-0.07,0.04,0.03,-0.01
E2,1.04,1.00,-0.97,0.74,0.67,0.52,-0.24,-0.19,0.15,-0.09,-0.06,0.02


## Reading the block means

Read *which* modules separate the groups before reading how far apart they are. `blue` holds 1,213
proteins and `yellow` 373; an eigenprotein over that much of the panel is close to the first
principal component of everything, so a split along it is one dominant axis rather than two
mechanisms. Dropping them, the "specific" space, did not help: prediction strength stayed below
threshold and the small cluster dissolved.

The interferon module is not part of the split, and that is the one informative thing here. It
separates the two descriptive groups by roughly 0.1 SD, essentially not at all, despite being the
module with the strongest clinical associations in step 04 and the one that cleanly separates
healthy volunteers in step 08. The interferon axis runs orthogonal to whatever k-means is
finding.

Put together with the result above: the partition that does not replicate is also not the
interferon axis. The module with the evidence behind it is the one this clustering cannot see.
That is the argument for clustering on the interferon module alone rather than for a better k.

The cell below locates the module by overlap with the curated interferon-stimulated gene (ISG) set, never by colour, weighted gene co-expression network analysis (WGCNA)
colours are assigned by module rank within a single fit and carry no meaning between fits.

In [5]:
ifn <- locate_ifn_module(mods, colnames(w$X))
cat(sprintf("interferon module '%s': %d proteins, %d of the %d curated ISG probes\n",
            ifn, sum(mods == ifn), attr(ifn, "hits")[[1]],
            length(read_ifn_panel(colnames(w$X)))))
if (ifn %in% colnames(blocks)) {
  cat("\nseparation between the descriptive groups, SD units:\n")
  print(round(blocks[, ifn, drop = FALSE], 2))
}

interferon module 'ivory': 14 proteins, 10 of the 23 curated ISG probes



separation between the descriptive groups, SD units:
   ivory
E1  0.04
E2 -0.09


In [6]:
saveRDS(list(endo = endo, K = K, descriptive_only = TRUE, space = CHOSEN_SPACE,
             S = S, spaces = SPACE, blocks = blocks, support = support,
             ps_threshold = PS_THRESHOLD, keep = keep, F = Fs, ifn = ifn),
        art("endotypes_A.rds"))

## Where this was run

Rendered notebooks are committed, so each records the machine, the R and the
package versions that produced its output.


In [7]:
run_provenance()

run on   : Annes-MacBook-Pro-193.local ( Darwin 27.0.0 )
date     : 2026-09-25 12:03 EDT 
R        : R version 4.4.3 (2025-02-28) | x86_64-apple-darwin13.4.0 
R comes from: /Users/adeslatt/miniforge3/envs/endotypes-proteomics 
packages :
   WGCNA            1.74
   ComplexHeatmap   2.22.0
   sva              3.54.0
   VarSelLCM        2.1.3.2
   cluster          2.1.8.1
   fpc              2.2.15
   circlize         0.4.18
